In [1]:
from pathlib import Path

print("Relevant Phase 5 files:\n")

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file() and (
        p.name in {
            "train.jsonl",
            "validation.jsonl",
            "policy.json",
            "localsql-phase5a-src.zip",
        }
        or "phase5" in str(p).lower()
    ):
        print(p)

Relevant Phase 5 files:

/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/validation.jsonl
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/train.jsonl
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/policy.json
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/PROJECT.md
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/.gitignore
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/pyproject.toml
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/README.md
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/uv.lock
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/.python-version
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/CLAUDE.md
/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/tes

In [2]:
from pathlib import Path
import shutil

BASE = Path(
    "/kaggle/input/datasets/hassanch6138/"
    "localsql-phase5-candidate"
)

SRC = BASE / "localsql-phase5a-src"
TRAIN = BASE / "train.jsonl"
VALIDATION = BASE / "validation.jsonl"
POLICY = BASE / "policy.json"

WORK = Path("/kaggle/working/localsql")

if WORK.exists():
    shutil.rmtree(WORK)

shutil.copytree(
    SRC,
    WORK,
    ignore=shutil.ignore_patterns("localsql-phase3-src"),
)

candidate_dir = WORK / "data" / "processed_phase5_candidate"
candidate_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(TRAIN, candidate_dir / "train.jsonl")
shutil.copy2(VALIDATION, candidate_dir / "validation.jsonl")
shutil.copy2(POLICY, candidate_dir / "policy.json")

print("Work:", WORK)
print("Runner:", (WORK / "scripts/run_qlora_smoke.py").exists())
print("Train:", (candidate_dir / "train.jsonl").exists())
print("Validation:", (candidate_dir / "validation.jsonl").exists())
print("Policy:", (candidate_dir / "policy.json").exists())
print("Old nested Phase 3 copied:", (WORK / "localsql-phase3-src").exists())

Work: /kaggle/working/localsql
Runner: True
Train: True
Validation: True
Policy: True
Old nested Phase 3 copied: False


In [3]:
from pathlib import Path
import hashlib

candidate_dir = Path(
    "/kaggle/working/localsql/data/processed_phase5_candidate"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()

for name in ["train.jsonl", "validation.jsonl", "policy.json"]:
    p = candidate_dir / name
    print(name, sha256(p))

print(
    "Train lines:",
    sum(1 for line in open(candidate_dir / "train.jsonl", encoding="utf-8")
        if line.strip())
)

print(
    "Validation lines:",
    sum(1 for line in open(candidate_dir / "validation.jsonl", encoding="utf-8")
        if line.strip())
)

train.jsonl E23A97EA746CEF24B17F6BEA8DC8440AB96313798837033EC76AF9CA79830196
validation.jsonl 441194A1CA027CF0E2B6D142879737D78D7B2459E0A4F28344EF7A55DDF48E68
policy.json 20756104604BCE2E660F53CF228E4858B3E5A271100C09489F3EC6EAE6694BC5
Train lines: 6067
Validation lines: 534


In [4]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [5]:
!python -m pip install -q -e . "transformers>=4.51" accelerate bitsandbytes peft trl

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 35.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.1 MB/s eta 0:00:00
  Building editable for localsql (pyproject.toml) ... done


In [6]:
import platform
import transformers

print("Python:", platform.python_version())
print("Transformers:", transformers.__version__)

Python: 3.12.13
Transformers: 5.0.0


In [7]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5-candidate-train-profile \
    --input data/processed_phase5_candidate/train.jsonl \
    --token-profile \
    --top-n-manifest 50

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 5.45MB/s]
tokenizer_config.json: 9.38kB [00:00, 36.4MB/s]
vocab.json: 2.78MB [00:00, 102MB/s]
merges.txt: 1.67MB [00:00, 144MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:01<00:00, 6.41MB/s]
Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Profiling 6067 examples from data/processed_phase5_candidate/train.jsonl ...
{
  "input_file": "data/processed_phase5_candidate/train.jsonl",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 6067,
  "prompt_only_token_stats": {
    "count": 6067,
    "min": 434.0,
    "median": 1796.0,
    "p90": 3328.0,
    "p95": 3705.0,
    "p99": 3849.0,
    "max": 3921.0
  },
  "prompt_only_counts_over_threshold": {
    ">3584": 316,
    ">4096": 0,
    ">8192": 0
  },
  "full_sft_sequence_token_stats": {
    "count": 6067,
    "min": 458.0,
    "median": 1842.0,
    "p90": 3383.0,
    "p95": 3735.0,
    "p99": 390

In [8]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5-candidate-validation-profile \
    --input data/processed_phase5_candidate/validation.jsonl \
    --token-profile \
    --top-n-manifest 50

Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Profiling 534 examples from data/processed_phase5_candidate/validation.jsonl ...
{
  "input_file": "data/processed_phase5_candidate/validation.jsonl",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 534,
  "prompt_only_token_stats": {
    "count": 534,
    "min": 506.0,
    "median": 1014.5,
    "p90": 3307.7,
    "p95": 3332.05,
    "p99": 3362.01,
    "max": 3396.0
  },
  "prompt_only_counts_over_threshold": {
    ">3584": 0,
    ">4096": 0,
    ">8192": 0
  },
  "full_sft_sequence_token_stats": {
    "count": 534,
    "min": 563.0,
    "median": 1088.5,
    "p90": 3367.7,
    "p95": 3400.35,
    "p99": 3463.69,
    "max": 3581.0
  },
  "full_sft_sequence_counts_over_threshold": {
    ">3584": 0,
    ">4096": 0,
    ">8192": 0
  },
  "prefix_boundary_check": {
    "description": "Verifies the full sequence's prefix equals the standalone prompt-only encoding for every example -

In [9]:
from pathlib import Path
import json

runs = Path("/kaggle/working/localsql/data/runs")

for run_id in [
    "phase5-candidate-train-profile",
    "phase5-candidate-validation-profile",
]:
    run = runs / run_id

    print("\n" + "=" * 80)
    print(run_id)
    print("=" * 80)

    for name in [
        "train_token_profile.json",
        "train_token_profile_longest_examples.json",
    ]:
        p = run / name

        print(f"\n--- {name} ---")

        if p.exists():
            with p.open("r", encoding="utf-8") as f:
                print(json.dumps(json.load(f), indent=2))
        else:
            print("MISSING")


phase5-candidate-train-profile

--- train_token_profile.json ---
{
  "input_file": "data/processed_phase5_candidate/train.jsonl",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 6067,
  "prompt_only_token_stats": {
    "count": 6067,
    "min": 434.0,
    "median": 1796.0,
    "p90": 3328.0,
    "p95": 3705.0,
    "p99": 3849.0,
    "max": 3921.0
  },
  "prompt_only_counts_over_threshold": {
    ">3584": 316,
    ">4096": 0,
    ">8192": 0
  },
  "full_sft_sequence_token_stats": {
    "count": 6067,
    "min": 458.0,
    "median": 1842.0,
    "p90": 3383.0,
    "p95": 3735.0,
    "p99": 3903.0,
    "max": 4005.0
  },
  "full_sft_sequence_counts_over_threshold": {
    ">3584": 331,
    ">4096": 0,
    ">8192": 0
  },
  "prefix_boundary_check": {
    "description": "Verifies the full sequence's prefix equals the standalone prompt-only encoding for every example -- the assumption completion-only masking depends on. Phase 4 validated this at 0/6,067 

In [10]:
from pathlib import Path
import json

runs = Path("/kaggle/working/localsql/data/runs")

files = [
    runs / "phase5-candidate-train-profile" / "train_token_profile.json",
    runs / "phase5-candidate-train-profile" / "train_longest_examples.json",
    runs / "phase5-candidate-validation-profile" / "validation_token_profile.json",
    runs / "phase5-candidate-validation-profile" / "validation_longest_examples.json",
]

for p in files:
    print("\n", "=" * 80)
    print(p)
    print("exists:", p.exists())
    if p.exists():
        with p.open("r", encoding="utf-8") as f:
            obj = json.load(f)

        if "longest" in p.name:
            print(json.dumps(obj, indent=2)[:12000])
        else:
            print(json.dumps(obj, indent=2))


/kaggle/working/localsql/data/runs/phase5-candidate-train-profile/train_token_profile.json
exists: True
{
  "input_file": "data/processed_phase5_candidate/train.jsonl",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 6067,
  "prompt_only_token_stats": {
    "count": 6067,
    "min": 434.0,
    "median": 1796.0,
    "p90": 3328.0,
    "p95": 3705.0,
    "p99": 3849.0,
    "max": 3921.0
  },
  "prompt_only_counts_over_threshold": {
    ">3584": 316,
    ">4096": 0,
    ">8192": 0
  },
  "full_sft_sequence_token_stats": {
    "count": 6067,
    "min": 458.0,
    "median": 1842.0,
    "p90": 3383.0,
    "p95": 3735.0,
    "p99": 3903.0,
    "max": 4005.0
  },
  "full_sft_sequence_counts_over_threshold": {
    ">3584": 331,
    ">4096": 0,
    ">8192": 0
  },
  "prefix_boundary_check": {
    "description": "Verifies the full sequence's prefix equals the standalone prompt-only encoding for every example -- the assumption completion-only masking depends

In [11]:
from pathlib import Path
import zipfile

out = Path("/kaggle/working/localsql-phase5-tokenizer-evidence.zip")

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        if p.exists():
            z.write(p, arcname=f"{p.parent.name}/{p.name}")

print(out)
print("exists:", out.exists())
print("size KB:", round(out.stat().st_size / 1024, 2))

/kaggle/working/localsql-phase5-tokenizer-evidence.zip
exists: True
size KB: 3.03


In [2]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("scripts/run_qlora_smoke.py"):
    if "phase5-b" in str(p).lower():
        print(p)

/kaggle/input/datasets/hassanch6138/phase5-b/scripts/run_qlora_smoke.py


In [3]:
from pathlib import Path
import shutil

SRC = Path("/kaggle/input/datasets/hassanch6138/phase5-b")
CANDIDATE = Path(
    "/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate"
)
WORK = Path("/kaggle/working/localsql")

# Start clean
if WORK.exists():
    shutil.rmtree(WORK)

# Copy Phase 5B source only.
# Exclude any old nested Phase 3 baggage if present.
shutil.copytree(
    SRC,
    WORK,
    ignore=shutil.ignore_patterns(
        "localsql-phase3-src",
        "__pycache__",
        ".pytest_cache",
    ),
)

# Copy the immutable Phase 5 candidate dataset
candidate_dir = WORK / "data" / "processed_phase5_candidate"
candidate_dir.mkdir(parents=True, exist_ok=True)

for name in ["train.jsonl", "validation.jsonl", "policy.json"]:
    shutil.copy2(
        CANDIDATE / name,
        candidate_dir / name,
    )

print("WORK:", WORK)
print("Runner:", (WORK / "scripts/run_qlora_smoke.py").exists())
print("Certification builder:", (WORK / "scripts/build_certification_sets.py").exists())
print("Checkpoint exporter:", (WORK / "scripts/export_checkpoint.py").exists())

print("Train:", (candidate_dir / "train.jsonl").exists())
print("Validation:", (candidate_dir / "validation.jsonl").exists())
print("Policy:", (candidate_dir / "policy.json").exists())

print("Old Phase 3 nested copy:",
      (WORK / "localsql-phase3-src").exists())

WORK: /kaggle/working/localsql
Runner: True
Certification builder: True
Checkpoint exporter: True
Train: True
Validation: True
Policy: True
Old Phase 3 nested copy: False


In [4]:
from pathlib import Path
import hashlib

candidate_dir = Path(
    "/kaggle/working/localsql/data/processed_phase5_candidate"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()

expected = {
    "train.jsonl":
        "E23A97EA746CEF24B17F6BEA8DC8440AB96313798837033EC76AF9CA79830196",
    "validation.jsonl":
        "441194A1CA027CF0E2B6D142879737D78D7B2459E0A4F28344EF7A55DDF48E68",
    "policy.json":
        "20756104604BCE2E660F53CF228E4858B3E5A271100C09489F3EC6EAE6694BC5",
}

for name, expected_hash in expected.items():
    actual = sha256(candidate_dir / name)
    print(name)
    print(" actual:  ", actual)
    print(" expected:", expected_hash)
    print(" MATCH:", actual == expected_hash)
    print()

train_lines = sum(
    1 for line in open(candidate_dir / "train.jsonl", encoding="utf-8")
    if line.strip()
)

val_lines = sum(
    1 for line in open(candidate_dir / "validation.jsonl", encoding="utf-8")
    if line.strip()
)

print("Train lines:", train_lines)
print("Validation lines:", val_lines)

train.jsonl
 actual:   E23A97EA746CEF24B17F6BEA8DC8440AB96313798837033EC76AF9CA79830196
 expected: E23A97EA746CEF24B17F6BEA8DC8440AB96313798837033EC76AF9CA79830196
 MATCH: True

validation.jsonl
 actual:   441194A1CA027CF0E2B6D142879737D78D7B2459E0A4F28344EF7A55DDF48E68
 expected: 441194A1CA027CF0E2B6D142879737D78D7B2459E0A4F28344EF7A55DDF48E68
 MATCH: True

policy.json
 actual:   20756104604BCE2E660F53CF228E4858B3E5A271100C09489F3EC6EAE6694BC5
 expected: 20756104604BCE2E660F53CF228E4858B3E5A271100C09489F3EC6EAE6694BC5
 MATCH: True

Train lines: 6067
Validation lines: 534


In [5]:
%cd /kaggle/working/localsql


/kaggle/working/localsql


In [6]:
!uv --version

uv 0.11.13 (x86_64-unknown-linux-gnu)


In [7]:
!python -m pip install -q uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 44.1 MB/s eta 0:00:0000:0100:01


In [8]:
!uv sync --group model --group train

Using CPython 3.11.16
Creating virtual environment at: .venv
Resolved 92 packages in 1ms
Prepared 86 packages in 41.06s                                           
░░░░░░░░░░░░░░░░░░░░ [0/86] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 86 packages in 52.41s                             
 + accelerate==1.15.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + bitsandbytes==0.50.2
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cuda-bindings==13.4.1
 + cuda-pathfinder==1.8.1
 + cuda-toolkit==13.0.3.0
 + datasets==5.0.1
 + dill==0.

In [9]:
import sys
import transformers
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cpu
Transformers: 5.0.0
CUDA available: False


In [10]:
!uv run python scripts/run_qlora_smoke.py \
    --run-id phase5-candidate-profile \
    --input data/processed_phase5_candidate/train.jsonl \
    --token-profile

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 998kB/s]
tokenizer_config.json: 100%|██████████████| 9.38k/9.38k [00:00<00:00, 14.8MB/s]
vocab.json: 100%|█████████████████████████| 2.78M/2.78M [00:00<00:00, 55.3MB/s]
merges.txt: 100%|█████████████████████████| 1.67M/1.67M [00:00<00:00, 68.5MB/s]

tokenizer.json: downloading bytes: ███████▏                | 3.40MB,  118kB/s  
tokenizer.json: downloading bytes: ████████████████████████| 3.40MB,  324kB/s  
tokenizer.json: reconstructing file: 100%|████████| 11.4MB / 11.4MB, 1.09MB/s  
Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Profiling 6067 examples from data/processed_phase5_candidate/train.jsonl ...
{
  "input_file": "data/processed_phase5_candidate/train.jsonl",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 6067,
  "prompt_only_token_stats": {
    "count": 6067,
    "min": 434.0,
    "median": 1796.0,
    "p90": 3328.0,
    "p95": 3705.0,
    "p

In [11]:
from pathlib import Path
import json
import hashlib
import statistics

p = Path(
    "/kaggle/working/localsql/data/runs/"
    "phase5-candidate-profile/train_token_lengths.jsonl"
)

print("Exists:", p.exists())

rows = [
    json.loads(line)
    for line in p.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

ids = [r["example_id"] for r in rows]
lengths = [r["full_sft_token_count"] for r in rows]

h = hashlib.sha256(p.read_bytes()).hexdigest()

print("Rows:", len(rows))
print("Unique example_ids:", len(set(ids)))
print("Min:", min(lengths))
print("Median:", statistics.median(lengths))
print("Max:", max(lengths))
print(">4096:", sum(x > 4096 for x in lengths))
print(
    "Prefix mismatches:",
    sum(not r["prefix_boundary_match"] for r in rows)
)
print("Manifest SHA256:", h)

Exists: True
Rows: 6067
Unique example_ids: 6067
Min: 458
Median: 1842
Max: 4005
>4096: 0
Prefix mismatches: 0
Manifest SHA256: df854c0d00232cbe7a46cf4727881286c8edf7440dbd0a3e3892fce8b2d44a22


In [12]:
!uv run python scripts/build_certification_sets.py --help

usage: build_certification_sets.py [-h] [--longest-n LONGEST_N]
                                   [--throughput-n THROUGHPUT_N]
                                   [--candidate-train CANDIDATE_TRAIN]
                                   [--longest-manifest LONGEST_MANIFEST]
                                   [--token-lengths-manifest TOKEN_LENGTHS_MANIFEST]
                                   [--out-dir OUT_DIR]

options:
  -h, --help            show this help message and exit
  --longest-n LONGEST_N
  --throughput-n THROUGHPUT_N
  --candidate-train CANDIDATE_TRAIN
  --longest-manifest LONGEST_MANIFEST
  --token-lengths-manifest TOKEN_LENGTHS_MANIFEST
  --out-dir OUT_DIR


In [13]:
!uv run python scripts/build_certification_sets.py \
    --candidate-train data/processed_phase5_candidate/train.jsonl \
    --longest-manifest data/runs/phase5-candidate-profile/train_longest_examples.json \
    --token-lengths-manifest data/runs/phase5-candidate-profile/train_token_lengths.jsonl \
    --out-dir data/certification \
    --longest-n 16 \
    --throughput-n 64

Longest-16 certification set: data/certification/longest_16.jsonl
  real length range: 3939.0 - 4005.0 tokens

Throughput sample (64 examples, REAL token lengths): data/certification/throughput_sample_64.jsonl
  {'count': 64, 'min': 458, 'median': 1841.0, 'p90': 3389.7, 'p95': 3715.2, 'p99': 3932.55, 'max': 4005, 'mean': 1935.58, 'representation_counts': {'compact_full_schema': 18, 'canonical_unchanged': 46}}


In [14]:
from pathlib import Path
import json
import hashlib

base = Path("/kaggle/working/localsql/data/certification")

files = [
    "longest_16.jsonl",
    "longest_16_report.json",
    "throughput_sample_64.jsonl",
    "throughput_sample_64_report.json",
]

for name in files:
    p = base / name

    print("\n" + "=" * 90)
    print(name)
    print("exists:", p.exists())

    if p.exists():
        print("sha256:", hashlib.sha256(p.read_bytes()).hexdigest())

        if p.suffix == ".json":
            obj = json.loads(p.read_text(encoding="utf-8"))
            print(json.dumps(obj, indent=2))
        else:
            rows = [
                json.loads(line)
                for line in p.read_text(encoding="utf-8").splitlines()
                if line.strip()
            ]
            print("rows:", len(rows))
            print("unique ids:", len({r["example_id"] for r in rows}))


longest_16.jsonl
exists: True
sha256: 3629414422bb9f07b0efd2f78ef20ff7a6eb6511ca1e0bcfd1d078203a11d6bf
rows: 16
unique ids: 16

longest_16_report.json
exists: True
sha256: 2323502b058c313a6034fc1a7d5a0aed21de2c8e76659bdd4495058b81ee6b87
{
  "task": "longest_n_certification_set",
  "n": 16,
  "example_ids_descending_length": [
    "birdsql/bird23-train-filtered:02059",
    "birdsql/bird23-train-filtered:06251",
    "birdsql/bird23-train-filtered:04593",
    "birdsql/bird23-train-filtered:02089",
    "birdsql/bird23-train-filtered:04595",
    "birdsql/bird23-train-filtered:02125",
    "birdsql/bird23-train-filtered:06252",
    "birdsql/bird23-train-filtered:04599",
    "birdsql/bird23-train-filtered:02096",
    "birdsql/bird23-train-filtered:02095",
    "birdsql/bird23-train-filtered:02092",
    "birdsql/bird23-train-filtered:02104",
    "birdsql/bird23-train-filtered:06248",
    "birdsql/bird23-train-filtered:04588",
    "birdsql/bird23-train-filtered:02088",
    "birdsql/bird23-train-

In [15]:
from pathlib import Path
import zipfile

ROOT = Path("/kaggle/working/localsql")

include = [
    ROOT / "data/runs/phase5-candidate-profile/train_token_profile.json",
    ROOT / "data/runs/phase5-candidate-profile/train_token_lengths.jsonl",
    ROOT / "data/runs/phase5-candidate-profile/train_longest_examples.json",

    ROOT / "data/certification/longest_16.jsonl",
    ROOT / "data/certification/longest_16_report.json",
    ROOT / "data/certification/throughput_sample_64.jsonl",
    ROOT / "data/certification/throughput_sample_64_report.json",

    ROOT / "data/processed_phase5_candidate/policy.json",
]

out = Path("/kaggle/working/localsql-phase5b-certification-evidence.zip")

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in include:
        assert p.exists(), f"Missing: {p}"
        z.write(p, arcname=str(p.relative_to(ROOT)))

print("Created:", out)
print("Size KB:", round(out.stat().st_size / 1024, 2))
print("\nContents:")
with zipfile.ZipFile(out) as z:
    for name in z.namelist():
        print(" ", name)

Created: /kaggle/working/localsql-phase5b-certification-evidence.zip
Size KB: 174.42

Contents:
  data/runs/phase5-candidate-profile/train_token_profile.json
  data/runs/phase5-candidate-profile/train_token_lengths.jsonl
  data/runs/phase5-candidate-profile/train_longest_examples.json
  data/certification/longest_16.jsonl
  data/certification/longest_16_report.json
  data/certification/throughput_sample_64.jsonl
  data/certification/throughput_sample_64_report.json
  data/processed_phase5_candidate/policy.json


In [ ]:
# GPU SESSION STARTED

In [1]:
import os

# Match Phase 4: use exactly one T4 even if Kaggle gives T4 x2.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("Total VRAM GiB:", round(props.total_memory / 1024**3, 3))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
Torch CUDA: 12.8
GPU: Tesla T4
Total VRAM GiB: 14.562


In [2]:
!python -m pip install -q \
    "transformers==5.0.0" \
    "accelerate==1.13.0" \
    "bitsandbytes==0.50.2" \
    "peft==0.19.1" \
    "trl==1.13.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00


In [3]:
import sys
import torch
import transformers
import accelerate
import bitsandbytes
import peft
import trl

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM GiB:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3)
)

Python: 3.12.13
Torch: 2.10.0+cu128
Torch CUDA: 12.8
Transformers: 5.0.0
Accelerate: 1.13.0
bitsandbytes: 0.50.2
PEFT: 0.19.1
TRL: 1.13.0
CUDA: True
GPU: Tesla T4
VRAM GiB: 14.562


In [4]:
from pathlib import Path
import shutil

SRC = Path("/kaggle/input/datasets/hassanch6138/phase5-b")
CANDIDATE = Path(
    "/kaggle/input/datasets/hassanch6138/localsql-phase5-candidate"
)
WORK = Path("/kaggle/working/localsql")

if WORK.exists():
    shutil.rmtree(WORK)

shutil.copytree(
    SRC,
    WORK,
    ignore=shutil.ignore_patterns(
        "localsql-phase3-src",
        "__pycache__",
        ".pytest_cache",
        ".venv",
    ),
)

candidate_dir = WORK / "data/processed_phase5_candidate"
candidate_dir.mkdir(parents=True, exist_ok=True)

for name in ["train.jsonl", "validation.jsonl", "policy.json"]:
    shutil.copy2(CANDIDATE / name, candidate_dir / name)

print("Repo copied:", WORK.exists())
print("Runner:", (WORK / "scripts/run_qlora_smoke.py").exists())
print("Candidate train:", (candidate_dir / "train.jsonl").exists())

Repo copied: True
Runner: True
Candidate train: True


In [5]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [6]:
!python -m pip install -q -e . --no-deps

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for localsql (pyproject.toml) ... done


In [7]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5-candidate-profile \
    --input data/processed_phase5_candidate/train.jsonl \
    --token-profile

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.31MB/s]
tokenizer_config.json: 9.38kB [00:00, 27.9MB/s]
vocab.json: 2.78MB [00:00, 71.7MB/s]
merges.txt: 1.67MB [00:00, 95.6MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 15.3MB/s]
Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Profiling 6067 examples from data/processed_phase5_candidate/train.jsonl ...
{
  "input_file": "data/processed_phase5_candidate/train.jsonl",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 6067,
  "prompt_only_token_stats": {
    "count": 6067,
    "min": 434.0,
    "median": 1796.0,
    "p90": 3328.0,
    "p95": 3705.0,
    "p99": 3849.0,
    "max": 3921.0
  },
  "prompt_only_counts_over_threshold": {
    ">3584": 316,
    ">4096": 0,
    ">8192": 0
  },
  "full_sft_sequence_token_stats": {
    "count": 6067,
    "min": 458.0,
    "median": 1842.0,
    "p90": 3383.0,
    "p95": 3735.0,
    "p99": 3

In [8]:
!python scripts/build_certification_sets.py \
    --candidate-train data/processed_phase5_candidate/train.jsonl \
    --longest-manifest data/runs/phase5-candidate-profile/train_longest_examples.json \
    --token-lengths-manifest data/runs/phase5-candidate-profile/train_token_lengths.jsonl \
    --out-dir data/certification \
    --longest-n 16 \
    --throughput-n 64

Longest-16 certification set: data/certification/longest_16.jsonl
  real length range: 3939.0 - 4005.0 tokens

Throughput sample (64 examples, REAL token lengths): data/certification/throughput_sample_64.jsonl
  {'count': 64, 'min': 458, 'median': 1841.0, 'p90': 3389.7, 'p95': 3715.2, 'p99': 3932.55, 'max': 4005, 'mean': 1935.58, 'representation_counts': {'compact_full_schema': 18, 'canonical_unchanged': 46}}


In [9]:
from pathlib import Path

for p in [
    Path("data/certification/longest_16.jsonl"),
    Path("data/certification/throughput_sample_64.jsonl"),
]:
    print(p, p.exists())

data/certification/longest_16.jsonl True
data/certification/throughput_sample_64.jsonl True


In [10]:
from pathlib import Path
import hashlib

expected = {
    "data/certification/longest_16.jsonl":
        "3629414422bb9f07b0efd2f78ef20ff7a6eb6511ca1e0bcfd1d078203a11d6bf",

    "data/certification/throughput_sample_64.jsonl":
        "321ccfbd9112106a4d91a26ee327482832e9e1d57780d8c23443110f7b99ef28",
}

for path, expected_hash in expected.items():
    actual = hashlib.sha256(Path(path).read_bytes()).hexdigest()

    print(path)
    print(" actual:  ", actual)
    print(" expected:", expected_hash)
    print(" MATCH:", actual == expected_hash)
    print()

data/certification/longest_16.jsonl
 actual:   3629414422bb9f07b0efd2f78ef20ff7a6eb6511ca1e0bcfd1d078203a11d6bf
 expected: 3629414422bb9f07b0efd2f78ef20ff7a6eb6511ca1e0bcfd1d078203a11d6bf
 MATCH: True

data/certification/throughput_sample_64.jsonl
 actual:   321ccfbd9112106a4d91a26ee327482832e9e1d57780d8c23443110f7b99ef28
 expected: 321ccfbd9112106a4d91a26ee327482832e9e1d57780d8c23443110f7b99ef28
 MATCH: True



In [12]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [14]:
!grep -n "train_file\|args.input\|--input" scripts/run_qlora_smoke.py

7:via `--input` (e.g. a Phase 5B certification set) -- never re-splits or
12:  --token-profile  Tokenizer-only profiling (see --input above). No 4-bit
28:        --input data/certification/longest_16.jsonl --max-steps 2
36:        --input data/certification/longest_16.jsonl --max-steps 2 \\
39:        --input data/certification/longest_16.jsonl --max-steps 4 \\
95:    train_path = repo_root / cfg.data.train_file
178:    (`input_path`, defaults to Phase 1's train.jsonl via `--input`).
318:        "train_file": str(train_path),
319:        "train_file_sha256": file_sha256(train_path),
427:        "train_file_sha256": run_config["train_file_sha256"],
483:        "--input",
487:        "(e.g. data/processed_phase5_candidate/train.jsonl). Defaults to configs/train.yaml's train_file.",
540:        if args.input:
541:            examples, input_path = load_arbitrary_jsonl_for_profiling(args.input, args.max_train_examples)


In [15]:
!grep -n "train_file" src/localsql/train/qlora_backend.py configs/train.yaml

configs/train.yaml:59:  train_file: data/processed/train.jsonl


In [13]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5b-mem-cert \
    --input data/certification/longest_16.jsonl \
    --max-steps 2 \
    --source-revision a834085

BLOCKER: training file not found: /kaggle/working/localsql/data/processed/train.jsonl
Run `uv run python scripts/prepare_bird.py` first (Phase 1).
